# Comparativo Qwen3-1.7B × Ministral-3-3B

Notebook para GPU NVIDIA. Execute um modelo por vez e mantenha os resultados no volume persistente.

In [ ]:
import gc, json, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'GPU CUDA não encontrada.'
gpu = torch.cuda.get_device_properties(0)
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', gpu.name, '| VRAM:', round(gpu.total_memory/1024**3, 2), 'GB')

## Localizar a raiz do projeto

In [ ]:
search_root = Path('/workspace') if Path('/workspace').exists() else Path.cwd()
configs = list(search_root.rglob('configs/model_candidates.yaml'))
assert configs, 'configs/model_candidates.yaml não encontrado. Envie o projeto completo.'
repo_dir = configs[0].parents[1]
benchmark = repo_dir / 'dataset/benchmark/expanded_100.jsonl'
assert benchmark.exists(), f'Benchmark ausente: {benchmark}'
print('Projeto:', repo_dir)
print('Benchmark:', benchmark)

In [ ]:
requirements = repo_dir / 'notebooks/pacote_teste_modelos/requirements_teste.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-r', str(requirements)], check=True)

## Executar o Qwen

O script retoma o JSONL existente. Use `--overwrite` somente se quiser apagar logicamente o progresso anterior.

In [ ]:
results_dir = repo_dir / 'evaluation/results/benchmark_100'
results_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'scripts.run_baseline', '--config', 'configs/model_candidates.yaml', '--model', 'qwen3-1.7b'], cwd=repo_dir, check=True)

## Executar o Ministral

Antes desta célula, reinicie o kernel se a VRAM do Qwen não tiver sido liberada.

In [ ]:
gc.collect(); torch.cuda.empty_cache()
subprocess.run([sys.executable, '-m', 'scripts.run_baseline', '--config', 'configs/model_candidates.yaml', '--model', 'ministral3-3b'], cwd=repo_dir, check=True)

## Gerar o CSV comparativo

In [ ]:
output_csv = results_dir / 'model_comparison_400.csv'
subprocess.run([sys.executable, '-m', 'evaluation.compare_models', '--results', str(results_dir), '--output', str(output_csv)], cwd=repo_dir, check=True)
print(output_csv.read_text(encoding='utf-8-sig'))